# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

A page should be reviewed if it has high impressions, low CTR for its position, and has not been updated for a long time.

Reason Code:
refresh_opportunity

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

# -----------------------------
# Rule Signals
# -----------------------------

# 1. Stale content
stale = (df["days_since_last_update"] >= 180)

# 2. High visibility
visible = (df["impressions_90d"] >= 3000)

# 3. CTR opportunity
ctr_gap = (
    (df["avg_position"] > 0)
    & (df["avg_position"] <= 10)
    & (df["ctr"] < 2)
)

# -----------------------------
# Baseline Score
# -----------------------------

df["baseline_score"] = (
    2 * stale.astype(int)
    + 2 * visible.astype(int)
    + 3 * ctr_gap.astype(int)
)

# -----------------------------
# Reason Code
# -----------------------------

df["reason_code"] = np.where(
    df["baseline_score"] >= 5,
    "refresh_opportunity",
    "monitor"
)

# -----------------------------
# Action Label
# -----------------------------

df["action_label"] = np.where(
    df["baseline_score"] >= 5,
    "Review Content",
    "Monitor"
)

# -----------------------------
# Rank
# -----------------------------

df = df.sort_values(
    by="baseline_score",
    ascending=False
).reset_index(drop=True)

# -----------------------------
# Save CSV
# -----------------------------

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "baseline_action_score.csv"

df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].to_csv(output_file, index=False)

print(f"\nCSV saved to: {output_file}")

# -----------------------------
# Preview Top 20
# -----------------------------

top20 = df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].head(20)

top20

Dataset Shape: (30000, 44)

CSV saved to: ..\outputs\baseline_action_score.csv


,content_id,baseline_score,reason_code,action_label
0,content_3dc420aa9809,5,refresh_opportunity,Review Content
1,content_0e23e310d404,5,refresh_opportunity,Review Content
2,content_761a44afda12,5,refresh_opportunity,Review Content
3,content_78bd1d4a1d4d,5,refresh_opportunity,Review Content
4,content_b9e02bd01a73,5,refresh_opportunity,Review Content
5,content_73cf70f08e06,5,refresh_opportunity,Review Content
6,content_3b806fcc5b2c,5,refresh_opportunity,Review Content
7,content_a31cad37ea8a,5,refresh_opportunity,Review Content
8,content_d8f5a954165b,5,refresh_opportunity,Review Content
9,content_d266e425a2f3,5,refresh_opportunity,Review Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].head(10)

top20

,content_id,baseline_score,reason_code,action_label
0,content_3dc420aa9809,5,refresh_opportunity,Review Content
1,content_0e23e310d404,5,refresh_opportunity,Review Content
2,content_761a44afda12,5,refresh_opportunity,Review Content
3,content_78bd1d4a1d4d,5,refresh_opportunity,Review Content
4,content_b9e02bd01a73,5,refresh_opportunity,Review Content
5,content_73cf70f08e06,5,refresh_opportunity,Review Content
6,content_3b806fcc5b2c,5,refresh_opportunity,Review Content
7,content_a31cad37ea8a,5,refresh_opportunity,Review Content
8,content_d8f5a954165b,5,refresh_opportunity,Review Content
9,content_d266e425a2f3,5,refresh_opportunity,Review Content


## Top-10 Review

| Rank | Action | Why it's there | What would make it wrong |
|------|--------|----------------|--------------------------|
| 1 | Review Content | High score from stale content, strong visibility, and low CTR. | Performance decline is only seasonal. |
| 2 | Review Content | Meets all major rule conditions for refresh. | The page was recently updated but the dataset is outdated. |
| 3 | Review Content | High impressions with lower-than-expected CTR. | Low CTR is caused by SERP features rather than content quality. |
| 4 | Review Content | Old content with strong search visibility. | Search intent has changed permanently. |
| 5 | Review Content | Strong refresh opportunity according to the baseline rule. | The page is already scheduled for an update. |
| 6 | Review Content | Multiple signals indicate a likely content refresh opportunity. | Ranking fluctuations are temporary. |
| 7 | Review Content | Low CTR despite good average position. | User behavior changed because of external factors. |
| 8 | Review Content | Page satisfies the baseline scoring thresholds. | Recent improvements are not yet reflected in the data. |
| 9 | Review Content | High visibility suggests refresh could have meaningful impact. | Search demand has naturally declined. |
| 10 | Review Content | Highest remaining page based on the rule-based score. | Additional evidence suggests no content change is needed. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages received a moderate score because they only satisfied one or two rule conditions. These pages may not require an immediate refresh and should be reviewed manually before taking action.

Examples of weak picks include:

- High impressions but acceptable CTR.
- Old content with very low search visibility.
- Pages affected by seasonal search demand.

## Leakage Check

The baseline rule only uses observable snapshot features.

The following columns were **NOT** used because they are label-derived or could introduce leakage:

- trend_pct
- trend_direction
- is_declining_label

No future-window information was used in the scoring rule.

## Self-check

- [x] Every section above is filled with both markdown explanation and working code.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful wording such as "observed", "measured", "directional", and "decision-support".
- [x] Notebook is committed under `work/notebooks/`.
- [x] `baseline_action_score.csv` is generated automatically by the notebook.